# 📊 MULTIHEIRTT - Analyse Simple

**Objectif**: Comprendre pourquoi certaines requêtes ont de bons résultats et d'autres non.

1. ✅ **15 meilleures requêtes** - Trouver les points communs
2. ❌ **15 pires requêtes** - Trouver les points communs
3. 🔍 **Comparer** - Trouver les différences clés

## 📦 Setup

In [25]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from collections import Counter
import re

# Configuration
DATA_DIR = Path("../data")
SUBMISSION_FILE = "./6_last_version/submission_final.csv"

QRELS_FILE = DATA_DIR / "MultiHeirtt_qrels.tsv"
QUERIES_FILE = DATA_DIR / "multiheirtt_queries.jsonl" / "queries.jsonl"
CORPUS_FILE = DATA_DIR / "multiheirtt_corpus.jsonl" / "corpus.jsonl"

# QRELS_FILE = DATA_DIR / "FinanceBench_qrels.tsv"
# QUERIES_FILE = DATA_DIR / "financebench_queries.jsonl" / "queries.jsonl"
# CORPUS_FILE = DATA_DIR / "financebench_corpus.jsonl" / "corpus.jsonl"


print("✅ Configuration chargée")

✅ Configuration chargée


## 📂 Charger les données

In [26]:
# Charger qrels
qrels_df = pd.read_csv(QRELS_FILE, sep='\t')
qrels = {}
for _, row in qrels_df.iterrows():
    if row['query_id'] not in qrels:
        qrels[row['query_id']] = {}
    qrels[row['query_id']][row['corpus_id']] = row['score']

# Charger queries
queries = {}
with open(QUERIES_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        q = json.loads(line)
        queries[q['_id']] = q['text']

# Charger corpus
corpus = {}
with open(CORPUS_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        doc = json.loads(line)
        corpus[doc['_id']] = {
            'title': doc.get('title', ''),
            'text': doc.get('text', '')
        }

# Charger résultats submission
sub_df = pd.read_csv(SUBMISSION_FILE)
results = {}
for _, row in sub_df.iterrows():
    if row['query_id'] not in results:
        results[row['query_id']] = []
    results[row['query_id']].append(row['corpus_id'])

print(f"✅ Chargé:")
print(f"   - {len(queries)} queries")
print(f"   - {len(corpus)} documents")
print(f"   - {len(results)} résultats")

✅ Chargé:
   - 974 queries
   - 10475 documents
   - 4671 résultats


## 📊 Calculer NDCG@10

In [27]:
def compute_ndcg(relevant_docs, retrieved_docs, k=10):
    """Calculer NDCG@k"""
    retrieved_k = retrieved_docs[:k]
    
    # DCG
    dcg = sum(relevant_docs.get(doc_id, 0) / np.log2(i + 2) 
              for i, doc_id in enumerate(retrieved_k))
    
    # IDCG
    ideal_scores = sorted(relevant_docs.values(), reverse=True)[:k]
    idcg = sum(score / np.log2(i + 2) for i, score in enumerate(ideal_scores))
    
    return dcg / idcg if idcg > 0 else 0.0

# Calculer NDCG pour chaque query
query_scores = []
for query_id in qrels.keys():
    if query_id in results:
        ndcg = compute_ndcg(qrels[query_id], results[query_id])
        query_scores.append({
            'query_id': query_id,
            'query_text': queries.get(query_id, ''),
            'ndcg_10': ndcg,
            'relevant_docs': list(qrels[query_id].keys()),
            'retrieved_docs': results[query_id][:10]
        })

df = pd.DataFrame(query_scores)
print(f"\n✅ NDCG@10 moyen: {df['ndcg_10'].mean():.4f}")
print(f"   Queries avec score 0: {(df['ndcg_10'] == 0).sum()} ({(df['ndcg_10'] == 0).mean()*100:.1f}%)")


✅ NDCG@10 moyen: 0.1571
   Queries avec score 0: 152 (52.1%)


---
## ❌ PARTIE 1: Analyser les 15 PIRES requêtes
### Trouver les points communs

In [28]:
# Sélectionner les 15 pires
worst_15 = df.nsmallest(15, 'ndcg_10')

print("="*80)
print("❌ 15 PIRES REQUÊTES")
print("="*80)

for idx, row in worst_15.iterrows():
    print(f"\n{'-'*70}")
    print(f"Query ID: {row['query_id']}")
    print(f"NDCG@10: {row['ndcg_10']:.4f}")
    print(f"Query: {row['query_text'][:100]}..." if len(row['query_text']) > 100 else f"Query: {row['query_text']}")
    print(f"\n📚 Docs pertinents ({len(row['relevant_docs'])}):")
    for doc_id in row['relevant_docs'][:3]:
        doc = corpus.get(doc_id, {})
        print(f"  - {doc_id}: {doc.get('title', 'N/A')[:60]}")
    
    print(f"\n🔍 Docs récupérés (5 premiers):")
    for doc_id in row['retrieved_docs'][:5]:
        is_rel = "✅" if doc_id in row['relevant_docs'] else "❌"
        doc = corpus.get(doc_id, {})
        print(f"  {is_rel} {doc_id}: {doc.get('title', 'N/A')[:60]}")

❌ 15 PIRES REQUÊTES

----------------------------------------------------------------------
Query ID: q8115b172
NDCG@10: 0.0000
Query: what is the pre-tax aggregate net unrealized loss in 2008?

📚 Docs pertinents (3):
  - d8115b2c6: 
  - d8115b370: 
  - d8115b3e8: 

🔍 Docs récupérés (5 premiers):
  ❌ d8a8d00de: 
  ❌ d8b3ef14a: 
  ❌ d873acb96: 
  ❌ d89393072: 
  ❌ d8e33b016: 

----------------------------------------------------------------------
Query ID: q81190e80
NDCG@10: 0.0000
Query: What's the sum of Debt maturities of Thereafter, and Capital lease obligations of Less than 1 year ?

📚 Docs pertinents (4):
  - d81139356: 
  - d811393e2: 
  - d81139478: 

🔍 Docs récupérés (5 premiers):
  ❌ d8843b0b6: 
  ❌ d82e92858: 
  ❌ d8e82ff9a: 
  ❌ d87dd6cd4: 
  ❌ d87c542e4: 

----------------------------------------------------------------------
Query ID: q811c3614
NDCG@10: 0.0000
Query: what was the percentage change in the allowance for loan losses from 2008 to 2009?

📚 Docs pertinents (1):


### 🔍 Analyser les caractéristiques communes des PIRES requêtes

In [29]:
def analyze_query_features(query_text):
    """Analyser les caractéristiques d'une requête"""
    text_lower = query_text.lower()
    
    return {
        'longueur_mots': len(query_text.split()),
        'contient_nombres': bool(re.search(r'\d', query_text)),
        'contient_annee': bool(re.search(r'\b(19|20)\d{2}\b', query_text)),
        'mots_calcul': any(w in text_lower for w in ['sum', 'total', 'average', 'calculate', 'ratio', 'percentage']),
        'mots_comparaison': any(w in text_lower for w in ['highest', 'lowest', 'most', 'least', 'greater', 'less']),
        'mots_temporel': any(w in text_lower for w in ['year', 'quarter', 'period', 'between', 'from', 'to']),
    }

def analyze_corpus_features(doc_ids):
    """Analyser les caractéristiques des documents"""
    texts = [corpus.get(doc_id, {}).get('text', '') for doc_id in doc_ids]
    
    lengths = [len(text) for text in texts]
    word_counts = [len(text.split()) for text in texts]
    num_counts = [len(re.findall(r'\b\d+(?:\.\d+)?\b', text)) for text in texts]
    year_counts = [len(re.findall(r'\b(19|20)\d{2}\b', text)) for text in texts]
    
    return {
        'nb_docs': len(doc_ids),
        'longueur_moy': np.mean(lengths) if lengths else 0,
        'mots_moy': np.mean(word_counts) if word_counts else 0,
        'nombres_moy': np.mean(num_counts) if num_counts else 0,
        'annees_moy': np.mean(year_counts) if year_counts else 0,
    }

# Analyser les pires requêtes
worst_query_features = []
worst_corpus_features = []

for _, row in worst_15.iterrows():
    worst_query_features.append(analyze_query_features(row['query_text']))
    worst_corpus_features.append(analyze_corpus_features(row['relevant_docs']))

worst_q_df = pd.DataFrame(worst_query_features)
worst_c_df = pd.DataFrame(worst_corpus_features)

print("="*80)
print("🔍 CARACTÉRISTIQUES COMMUNES DES PIRES REQUÊTES")
print("="*80)
print("\n📝 REQUÊTES:")
print(worst_q_df.mean().to_string())
print("\n📚 CORPUS PERTINENTS:")
print(worst_c_df.mean().to_string())

🔍 CARACTÉRISTIQUES COMMUNES DES PIRES REQUÊTES

📝 REQUÊTES:
longueur_mots       15.933333
contient_nombres     0.666667
contient_annee       0.533333
mots_calcul          0.533333
mots_comparaison     0.333333
mots_temporel        0.666667

📚 CORPUS PERTINENTS:
nb_docs            3.733333
longueur_moy    4899.398413
mots_moy         819.733333
nombres_moy      123.810317
annees_moy        18.335714


---
## ✅ PARTIE 2: Analyser les 15 MEILLEURES requêtes
### Trouver les points communs

In [30]:
# Sélectionner les 15 meilleures
best_15 = df.nlargest(15, 'ndcg_10')

print("="*80)
print("✅ 15 MEILLEURES REQUÊTES")
print("="*80)

for idx, row in best_15.iterrows():
    print(f"\n{'-'*70}")
    print(f"Query ID: {row['query_id']}")
    print(f"NDCG@10: {row['ndcg_10']:.4f}")
    print(f"Query: {row['query_text'][:100]}..." if len(row['query_text']) > 100 else f"Query: {row['query_text']}")
    print(f"\n📚 Docs pertinents ({len(row['relevant_docs'])}):")
    for doc_id in row['relevant_docs'][:3]:
        doc = corpus.get(doc_id, {})
        print(f"  - {doc_id}: {doc.get('title', 'N/A')[:60]}")
    
    print(f"\n🔍 Docs récupérés (5 premiers):")
    for doc_id in row['retrieved_docs'][:5]:
        is_rel = "✅" if doc_id in row['relevant_docs'] else "❌"
        doc = corpus.get(doc_id, {})
        print(f"  {is_rel} {doc_id}: {doc.get('title', 'N/A')[:60]}")

✅ 15 MEILLEURES REQUÊTES

----------------------------------------------------------------------
Query ID: q83554574
NDCG@10: 0.7095
Query: In the year with largest amount of SquareFeet of Total Consolidated Portfolio, what's the increasing...

📚 Docs pertinents (4):
  - d82ba7170: 
  - d82ba71d4: 
  - d82ba7260: 

🔍 Docs récupérés (5 premiers):
  ✅ d82ba7300: 
  ❌ d855e7660: 
  ❌ d822356d2: 
  ✅ d82ba7170: 
  ✅ d82ba71d4: 

----------------------------------------------------------------------
Query ID: q813a3eca
NDCG@10: 0.6367
Query: What is the sum of CET1 capital, Tier 1 capital and Total capital in 2017? (in million)

📚 Docs pertinents (4):
  - d813a3fec: 
  - d813a40b4: 
  - d813a4104: 

🔍 Docs récupérés (5 premiers):
  ✅ d813a41c2: 
  ✅ d813a4104: 
  ❌ d88d47b96: 
  ❌ d8c773afe: 
  ❌ d8df72a10: 

----------------------------------------------------------------------
Query ID: q825e6902
NDCG@10: 0.6367
Query: what is the annualized return for the investment in the allegion plc d

### 🔍 Analyser les caractéristiques communes des MEILLEURES requêtes

In [31]:
# Analyser les meilleures requêtes
best_query_features = []
best_corpus_features = []

for _, row in best_15.iterrows():
    best_query_features.append(analyze_query_features(row['query_text']))
    best_corpus_features.append(analyze_corpus_features(row['relevant_docs']))

best_q_df = pd.DataFrame(best_query_features)
best_c_df = pd.DataFrame(best_corpus_features)

print("="*80)
print("🔍 CARACTÉRISTIQUES COMMUNES DES MEILLEURES REQUÊTES")
print("="*80)
print("\n📝 REQUÊTES:")
print(best_q_df.mean().to_string())
print("\n📚 CORPUS PERTINENTS:")
print(best_c_df.mean().to_string())

🔍 CARACTÉRISTIQUES COMMUNES DES MEILLEURES REQUÊTES

📝 REQUÊTES:
longueur_mots       19.133333
contient_nombres     0.733333
contient_annee       0.666667
mots_calcul          0.733333
mots_comparaison     0.200000
mots_temporel        0.800000

📚 CORPUS PERTINENTS:
nb_docs            3.000000
longueur_moy    4682.827778
mots_moy         789.633333
nombres_moy      127.877778
annees_moy        24.988889


---
## 🔍 PARTIE 3: COMPARAISON - Trouver les différences clés
### Qu'est-ce qui distingue les bonnes requêtes des mauvaises ?

In [8]:
print("="*80)
print("🔍 COMPARAISON: MEILLEURES vs PIRES")
print("="*80)

print("\n📝 CARACTÉRISTIQUES DES REQUÊTES:")
print("="*80)
comparison_q = pd.DataFrame({
    'PIRES': worst_q_df.mean(),
    'MEILLEURES': best_q_df.mean(),
    'DIFFÉRENCE': best_q_df.mean() - worst_q_df.mean()
})
print(comparison_q.to_string())

print("\n\n📚 CARACTÉRISTIQUES DU CORPUS:")
print("="*80)
comparison_c = pd.DataFrame({
    'PIRES': worst_c_df.mean(),
    'MEILLEURES': best_c_df.mean(),
    'DIFFÉRENCE': best_c_df.mean() - worst_c_df.mean()
})
print(comparison_c.to_string())

🔍 COMPARAISON: MEILLEURES vs PIRES

📝 CARACTÉRISTIQUES DES REQUÊTES:
                      PIRES  MEILLEURES  DIFFÉRENCE
longueur_mots     15.933333   19.133333    3.200000
contient_nombres   0.666667    0.733333    0.066667
contient_annee     0.533333    0.666667    0.133333
mots_calcul        0.533333    0.733333    0.200000
mots_comparaison   0.333333    0.200000   -0.133333
mots_temporel      0.666667    0.800000    0.133333


📚 CARACTÉRISTIQUES DU CORPUS:
                    PIRES   MEILLEURES  DIFFÉRENCE
nb_docs          3.733333     3.000000   -0.733333
longueur_moy  4899.398413  4682.827778 -216.570635
mots_moy       819.733333   789.633333  -30.100000
nombres_moy    123.810317   127.877778    4.067460
annees_moy      18.335714    24.988889    6.653175


### 💡 Insights et Recommandations

In [9]:
print("="*80)
print("💡 INSIGHTS CLÉS")
print("="*80)

# NDCG scores
worst_avg_ndcg = worst_15['ndcg_10'].mean()
best_avg_ndcg = best_15['ndcg_10'].mean()

print(f"\n📊 SCORES:")
print(f"   PIRES:      NDCG@10 = {worst_avg_ndcg:.4f}")
print(f"   MEILLEURES: NDCG@10 = {best_avg_ndcg:.4f}")
print(f"   ÉCART:      {best_avg_ndcg - worst_avg_ndcg:.4f}")

print(f"\n🔍 DIFFÉRENCES PRINCIPALES:")

# Query differences
if comparison_q.loc['longueur_mots', 'DIFFÉRENCE'] != 0:
    direction = "plus longues" if comparison_q.loc['longueur_mots', 'DIFFÉRENCE'] > 0 else "plus courtes"
    print(f"\n📝 REQUÊTES:")
    print(f"   - Les bonnes requêtes sont {direction} ({comparison_q.loc['longueur_mots', 'DIFFÉRENCE']:.1f} mots de différence)")

# Corpus differences
print(f"\n📚 CORPUS:")
if comparison_c.loc['nb_docs', 'DIFFÉRENCE'] != 0:
    direction = "plus" if comparison_c.loc['nb_docs', 'DIFFÉRENCE'] > 0 else "moins"
    print(f"   - Les bonnes requêtes ont besoin de {direction} de documents pertinents ({comparison_c.loc['nb_docs', 'DIFFÉRENCE']:.1f})")

if comparison_c.loc['longueur_moy', 'DIFFÉRENCE'] != 0:
    direction = "plus longs" if comparison_c.loc['longueur_moy', 'DIFFÉRENCE'] > 0 else "plus courts"
    print(f"   - Les docs pertinents sont {direction} ({comparison_c.loc['longueur_moy', 'DIFFÉRENCE']:.0f} caractères)")

if comparison_c.loc['nombres_moy', 'DIFFÉRENCE'] != 0:
    direction = "plus" if comparison_c.loc['nombres_moy', 'DIFFÉRENCE'] > 0 else "moins"
    print(f"   - Les docs pertinents contiennent {direction} de nombres ({comparison_c.loc['nombres_moy', 'DIFFÉRENCE']:.1f})")

print(f"\n\n💡 RECOMMANDATIONS:")
print(f"   1. Améliorer la récupération pour les requêtes complexes")
print(f"   2. Optimiser la correspondance numérique (années, montants)")
print(f"   3. Améliorer le ranking pour les requêtes multi-documents")
print(f"   4. Augmenter le top_k pour récupérer plus de documents pertinents")

💡 INSIGHTS CLÉS

📊 SCORES:
   PIRES:      NDCG@10 = 0.0000
   MEILLEURES: NDCG@10 = 0.6219
   ÉCART:      0.6219

🔍 DIFFÉRENCES PRINCIPALES:

📝 REQUÊTES:
   - Les bonnes requêtes sont plus longues (3.2 mots de différence)

📚 CORPUS:
   - Les bonnes requêtes ont besoin de moins de documents pertinents (-0.7)
   - Les docs pertinents sont plus courts (-217 caractères)
   - Les docs pertinents contiennent plus de nombres (4.1)


💡 RECOMMANDATIONS:
   1. Améliorer la récupération pour les requêtes complexes
   2. Optimiser la correspondance numérique (années, montants)
   3. Améliorer le ranking pour les requêtes multi-documents
   4. Augmenter le top_k pour récupérer plus de documents pertinents


---
## 📊 Résumé Visual

In [10]:
# Créer un résumé simple
summary = pd.DataFrame({
    'Métrique': [
        'NDCG@10 moyen',
        'Longueur requête (mots)',
        'Contient calcul (%)',
        'Nb docs pertinents',
        'Longueur doc (chars)',
        'Nombres par doc'
    ],
    '❌ PIRES': [
        f"{worst_avg_ndcg:.4f}",
        f"{worst_q_df['longueur_mots'].mean():.1f}",
        f"{worst_q_df['mots_calcul'].mean()*100:.0f}%",
        f"{worst_c_df['nb_docs'].mean():.1f}",
        f"{worst_c_df['longueur_moy'].mean():.0f}",
        f"{worst_c_df['nombres_moy'].mean():.1f}"
    ],
    '✅ MEILLEURES': [
        f"{best_avg_ndcg:.4f}",
        f"{best_q_df['longueur_mots'].mean():.1f}",
        f"{best_q_df['mots_calcul'].mean()*100:.0f}%",
        f"{best_c_df['nb_docs'].mean():.1f}",
        f"{best_c_df['longueur_moy'].mean():.0f}",
        f"{best_c_df['nombres_moy'].mean():.1f}"
    ]
})

print("="*80)
print("📊 RÉSUMÉ COMPARATIF")
print("="*80)
print(summary.to_string(index=False))

print("\n" + "="*80)
print("✅ ANALYSE TERMINÉE !")
print("="*80)

📊 RÉSUMÉ COMPARATIF
               Métrique ❌ PIRES ✅ MEILLEURES
          NDCG@10 moyen  0.0000       0.6219
Longueur requête (mots)    15.9         19.1
    Contient calcul (%)     53%          73%
     Nb docs pertinents     3.7          3.0
   Longueur doc (chars)    4899         4683
        Nombres par doc   123.8        127.9

✅ ANALYSE TERMINÉE !
